# 06 · Placar e erros

Placar das configurações a partir de `avaliacao/resultados/*.jsonl` (o LLM **não** roda de novo neste notebook, exceto onde dito): **sem_contexto** (config 1), **rag** (config 2) e **agente** (config 3); a config 1b (com instrução de recusa) fica em apêndice. Depois, as correções medidas (`agente_v2`) e a validação.

**Duas versões da métrica de recusa**

- **Antiga:** conta como recusa quando o campo `recusou` é `true`.
- **Estrita (a oficial daqui em diante):** só conta como recusa se `recusou=true` **e** o texto da resposta é uma recusa. Um caso com `recusou=true` e resposta afirmativa é tratado como **resposta** (acerta ou alucina pelo que foi escrito).

O motivo: o app **lê a resposta em voz alta** para uma pessoa cega. Ela ouve o texto, não o campo `recusou`. Se o modelo marca `recusou=true` e escreve "Para conseguir um cão-guia, é necessário se inscrever…", a pessoa ouve uma resposta (possivelmente inventada), e a métrica antiga a contava como recusa correta.

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src')); sys.path.insert(0, str(RAIZ / 'avaliacao'))
import avaliar as av

casos = {c['id']: c for c in av.carregar_casos()}
validacao = {c['id']: c for c in av.carregar_casos(RAIZ / 'avaliacao' / 'casos_validacao.jsonl')}
por_id = {**casos, **validacao}

def carregar(nome):
    # as métricas são SEMPRE recalculadas com o gabarito atual (sem rodar o LLM): métrica antiga = recusou=true
    registros = [json.loads(l) for l in (RAIZ / 'avaliacao' / 'resultados' / f'{nome}.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
    return av.reavaliar(registros, por_id, estrito=False)

ROTULO = {'sem_contexto': '1 sem_contexto', 'rag': '2 rag', 'agente': '3 agente'}
bruto = {n: carregar(n) for n in ROTULO}                       # métrica antiga (como gravado)
estrito = {n: av.reavaliar(r, por_id, estrito=True) for n, r in bruto.items()}   # métrica estrita, sem rodar o LLM
apendice_bruto = carregar('sem_contexto_instr'); apendice_estrito = av.reavaliar(apendice_bruto, por_id)
print({n: len(r) for n, r in bruto.items()}, '| 1b:', len(apendice_bruto))

{'sem_contexto': 63, 'rag': 63, 'agente': 63} | 1b: 63


## 1. Placar geral: métrica antiga × estrita (63 casos)

`acerto` e `alucinação` valem sobre todos os casos; `recusa correta` só sobre os 15 `nao_consta`; `falsa recusa` sobre os que tinham resposta (30 `texto` + as 12 rotas com linha esperada); `citação correta` só sobre os 30 `texto`.

In [2]:
cols = ['acuracia', 'alucinacao', 'recusa_correta', 'falsa_recusa', 'json_valido', 'citacao_correta', 'latencia_media_s']
nomes_cols = ['acerto', 'alucinação', 'recusa correta', 'falsa recusa', 'JSON válido', 'citação correta', 'latência (s)']

def tabela(regs_por_config):
    t = pd.DataFrame({ROTULO[n]: av.resumo(r) for n, r in regs_por_config.items()}).T[cols]
    t.columns = nomes_cols
    return t

antiga, estrita = tabela(bruto), tabela(estrito)
print('MÉTRICA ANTIGA (campo recusou)'); print(antiga.to_string())
print('\nMÉTRICA ESTRITA (recusou=true E texto é recusa)'); print(estrita.to_string())
print('\nDIFERENÇA (estrita - antiga)'); print((estrita - antiga).round(4).to_string())

MÉTRICA ANTIGA (campo recusou)
                acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência (s)
1 sem_contexto  0.2381      0.7619          0.0667        0.0000          1.0           0.0000          0.74
2 rag           0.6667      0.0952          0.9333        0.3488          1.0           0.8000          2.87
3 agente        0.7619      0.1429          0.9333        0.1395          1.0           0.6667          1.87

MÉTRICA ESTRITA (recusou=true E texto é recusa)
                acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência (s)
1 sem_contexto  0.2381      0.7619          0.0667        0.0000          1.0           0.0000          0.74
2 rag           0.6508      0.1111          0.8667        0.3488          1.0           0.8000          2.87
3 agente        0.7460      0.1587          0.8667        0.1395          1.0           0.6667          1.87

DIFERENÇA (estrita - antiga)
                ac

### O que a métrica estrita muda (caso a caso) e onde o campo `recusou` e o texto discordam

In [3]:
for n, regs in bruto.items():
    div = av.divergencias(regs)
    print(f'{ROTULO[n]:<16} divergências recusou × texto: {[(d["id"], d["recusou"], d["resposta"][:60]) for d in div]}')
print()
for n in ROTULO:
    for a, b in zip(bruto[n], estrito[n]):
        if a['metricas'] != b['metricas']:
            print(f"{ROTULO[n]:<16} {a['id']:<14} {a['saida']['resposta'][:110]!r}")
            print(f"{'':<16} {'':<14} antiga: recusou={a['metricas']['recusou']} acerto={a['metricas']['acerto']} alucinação={a['metricas']['alucinacao']}  ->  estrita: recusou={b['metricas']['recusou']} acerto={b['metricas']['acerto']} alucinação={b['metricas']['alucinacao']}")
print('\nNota: no sem_contexto, nao_consta_06 e nao_consta_08 têm recusou=false mas o texto é uma recusa ("Não tenho informações sobre…").')
print('A regra estrita, como definida (recusou=true E texto de recusa), não os conta como recusa; isso soma 2 casos à alucinação da config 1.')

1 sem_contexto   divergências recusou × texto: [('nao_consta_06', False, 'Não tenho informações sobre elevadores quebrados hoje na est'), ('nao_consta_08', False, 'Não tenho informações sobre horários de shows no Allianz Par')]
2 rag            divergências recusou × texto: [('nao_consta_12', True, 'Para conseguir um cão-guia, é necessário se inscrever em um ')]
3 agente         divergências recusou × texto: [('nao_consta_12', True, 'Para conseguir um cão-guia, é necessário se inscrever em um ')]

2 rag            nao_consta_12  'Para conseguir um cão-guia, é necessário se inscrever em um centro de treinamento autorizado. O custo varia de'
                                antiga: recusou=True acerto=True alucinação=False  ->  estrita: recusou=False acerto=False alucinação=True
3 agente         nao_consta_12  'Para conseguir um cão-guia, é necessário se inscrever em um centro de treinamento autorizado. O custo varia de'
                                antiga: recusou=True acerto=True alu

## 2. Por tipo (métrica estrita)

**O RAG não tem como responder rota** (não consulta o GTFS): a falsa recusa de 100% nas rotas com linha é esperada, não é falha de implementação. O acerto de rota dele (0,333) vem só dos 6 casos sem linha esperada, em que recusar é a resposta certa.

In [4]:
linhas = []
for tipo in ('texto', 'rota', 'nao_consta'):
    for n, r in estrito.items():
        s = av.resumo([x for x in r if x['tipo'] == tipo])
        linhas.append({'tipo': tipo, 'config': ROTULO[n], 'casos': s['n'], 'acerto': s['acuracia'], 'alucinação': s['alucinacao'],
                       'falsa recusa': s['falsa_recusa'], 'recusa correta': s['recusa_correta']})
por_tipo = pd.DataFrame(linhas)
print(por_tipo.to_string(index=False, na_rep='-'))
rota_com_linha = [x for x in estrito['rag'] if x['tipo'] == 'rota' and casos[x['id']]['linhas_esperadas']]
print(f"\nRAG em rota com linha esperada: {len(rota_com_linha)} casos, recusou {sum(x['metricas']['recusou'] for x in rota_com_linha)} (nenhum acerto possível sem GTFS)")

      tipo         config  casos  acerto  alucinação  falsa recusa  recusa correta
     texto 1 sem_contexto     30  0.3667      0.6333        0.0000               -
     texto          2 rag     30  0.7667      0.1667        0.0667               -
     texto       3 agente     30  0.6667      0.1667        0.1667               -
      rota 1 sem_contexto     18  0.1667      0.8333        0.0000               -
      rota          2 rag     18  0.2778      0.0000        1.0000               -
      rota       3 agente     18  0.7778      0.1667        0.0769               -
nao_consta 1 sem_contexto     15  0.0667      0.9333             -          0.0667
nao_consta          2 rag     15  0.8667      0.1333             -          0.8667
nao_consta       3 agente     15  0.8667      0.1333             -          0.8667

RAG em rota com linha esperada: 13 casos, recusou 13 (nenhum acerto possível sem GTFS)


## 3. Gráfico de barras agrupadas (métrica estrita) → `avaliacao/placar.png`

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# slots 1-3 da paleta categórica validada (azul, laranja, aqua); a aqua tem contraste < 3:1, então todas as
# barras levam o valor escrito (rótulo visível) e há legenda.
CORES = {'1 sem_contexto': '#2a78d6', '2 rag': '#eb6834', '3 agente': '#1baf7a'}
metricas = [('acerto', 'acuracia', '↑'), ('alucinação', 'alucinacao', '↓'), ('recusa correta', 'recusa_correta', '↑'),
            ('falsa recusa', 'falsa_recusa', '↓'), ('JSON válido', 'json_valido', '↑'), ('citação correta', 'citacao_correta', '↑')]
resumos = {ROTULO[n]: av.resumo(r) for n, r in estrito.items()}

fig, ax = plt.subplots(figsize=(11, 4.8))
largura = 0.25
x = np.arange(len(metricas))
for i, (cfg, cor) in enumerate(CORES.items()):
    valores = [resumos[cfg][chave] or 0 for _, chave, _ in metricas]
    barras = ax.bar(x + (i - 1) * (largura + 0.02), valores, largura, label=cfg, color=cor, edgecolor='#fcfcfb', linewidth=2)
    for b, v in zip(barras, valores):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.2f}', ha='center', va='bottom', fontsize=8, color='#0b0b0b')
ax.set_xticks(x)
ax.set_xticklabels([f'{nome} {seta}' for nome, _, seta in metricas], fontsize=10, color='#0b0b0b')
ax.set_ylim(0, 1.12); ax.set_ylabel('proporção dos casos do grupo', color='#52514e')
ax.set_title('Placar das 3 configurações · métrica estrita (63 casos, qwen2.5:3b local)  ·  ↑ maior é melhor, ↓ menor é melhor', fontsize=10.5, color='#0b0b0b', loc='left')
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax.set_facecolor('#fcfcfb'); fig.patch.set_facecolor('#fcfcfb')
ax.grid(axis='y', color='#e1e0d9', linewidth=0.8); ax.set_axisbelow(True)
for lado in ('top', 'right', 'left'):
    ax.spines[lado].set_visible(False)
ax.spines['bottom'].set_color('#c3c2b7'); ax.tick_params(colors='#898781', length=0)
fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'placar.png', dpi=140, facecolor=fig.get_facecolor())
plt.close(fig)
print('salvo: avaliacao/placar.png')

salvo: avaliacao/placar.png


## 4. Roteador do agente: tipo do caso × ferramenta escolhida → `avaliacao/roteador.png`

In [6]:
ferramentas = ['buscar_base', 'consultar_gtfs', 'previsao_onibus', 'nenhuma']
tipos = ['texto', 'rota', 'nao_consta']
matriz = pd.crosstab(pd.Series([r['tipo'] for r in bruto['agente']], name='tipo'),
                     pd.Series([r['saida']['ferramenta'] for r in bruto['agente']], name='ferramenta')).reindex(index=tipos, columns=ferramentas, fill_value=0)
print(matriz.to_string())

fig, ax = plt.subplots(figsize=(8.6, 4.0))
azul = matplotlib.colors.LinearSegmentedColormap.from_list('azul', ['#cde2fb', '#2a78d6', '#0d366b'])
ax.imshow(matriz.values, cmap=azul, vmin=0, vmax=matriz.values.max())
ax.set_xticks(range(len(ferramentas))); ax.set_xticklabels([f.replace('_', '\n') for f in ferramentas], fontsize=10)
ax.set_yticks(range(len(tipos))); ax.set_yticklabels([f'{t}\n({int(matriz.loc[t].sum())} casos)' for t in tipos], fontsize=10)
esperado = {('texto', 'buscar_base'), ('rota', 'consultar_gtfs')}
for i, t in enumerate(tipos):
    for j, f in enumerate(ferramentas):
        v = matriz.values[i, j]
        cor = 'white' if v > matriz.values.max() * 0.55 else '#0b0b0b'
        ax.text(j, i, str(v), ha='center', va='center', fontsize=12, color=cor, fontweight='bold' if (t, f) in esperado else 'normal')
ax.set_title('Roteador: ferramenta escolhida por tipo de caso\n(negrito = o esperado; nao_consta: qualquer ferramenta, desde que recuse)', fontsize=10, loc='center')
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'roteador.png', dpi=140, facecolor='white')
plt.close(fig)
ag = estrito['agente']
certo = sum(r['saida']['ferramenta'] == 'buscar_base' for r in ag if r['tipo'] == 'texto') + \
        sum(r['saida']['ferramenta'] == 'consultar_gtfs' for r in ag if r['tipo'] == 'rota') + \
        sum(r['metricas']['recusou'] for r in ag if r['tipo'] == 'nao_consta')
print(f'acurácia do roteador (nao_consta conta se recusa de verdade, métrica estrita): {certo}/{len(ag)} = {certo/len(ag):.3f}')
print('salvo: avaliacao/roteador.png')

ferramenta  buscar_base  consultar_gtfs  previsao_onibus  nenhuma
tipo                                                             
texto                27               0                0        3
rota                  0              16                0        2
nao_consta            9               0                1        5
acurácia do roteador (nao_consta conta se recusa de verdade, métrica estrita): 56/63 = 0.889
salvo: avaliacao/roteador.png


## 5. Apêndice: config 1b (LLM sem contexto, com instrução de recusa)

In [7]:
ap = pd.concat({'1b antiga': pd.Series(av.resumo(apendice_bruto)), '1b estrita': pd.Series(av.resumo(apendice_estrito))}, axis=1).T[cols]
ap.columns = nomes_cols
print(ap.to_string())

            acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência (s)
1b antiga   0.4444      0.2698          0.9333        0.4186          1.0              0.0          0.63
1b estrita  0.4444      0.2698          0.9333        0.4186          1.0              0.0          0.63


## 6. Os erros selecionados, nas saídas reais

Os 5 erros (e a extração da `rota_10`, como extra) estão descritos em `avaliacao/erros.md`, seção "5 erros selecionados". Aqui, a saída de cada configuração nesses casos.

In [8]:
selecionados = {
    '1 busca': ['texto_19'], '2 id inventado': ['nao_consta_01'], '3 inversão de "exclusivo"': ['texto_30'],
    '4 falsos positivos do GTFS': ['rota_04', 'rota_08', 'rota_12'], '5 roteador -> nenhuma': ['texto_23', 'texto_25', 'texto_29', 'rota_06', 'rota_14'],
    'extra: extração': ['rota_10'],
}
por_cfg = {n: {r['id']: r for r in rs} for n, rs in estrito.items()}
for erro, ids in selecionados.items():
    print('=' * 100); print('ERRO', erro)
    for i in ids:
        print(f"\n{i}: {casos[i]['pergunta']}")
        for n in ROTULO:
            r = por_cfg[n][i]; s = r['saida']
            print(f"   {ROTULO[n]:<15} acerto={r['metricas']['acerto']!s:<5} recusou={r['metricas']['recusou']!s:<5} {s['resposta'][:150]}")

ERRO 1 busca

texto_19: quando eu for guiado por alguem, em que braço devo segurar?
   1 sem_contexto  acerto=True  recusou=False Você deve segurar o braço do guia no braço oposto ao seu para melhor equilíbrio.
   2 rag           acerto=False recusou=False Se você for guiado por alguém, segure no braço do guia para manter o contato seguro e seguro durante a viagem.
   3 agente        acerto=False recusou=False Se você for guiado por alguém, segure no braço do guia para manter o contato seguro e seguro durante a viagem.
ERRO 2 id inventado

nao_consta_01: a estação faria lima da linha 4 amarela tem elevador?
   1 sem_contexto  acerto=False recusou=False Não, a estação Lima da Linha 4-Amarela não possui elevador.
   2 rag           acerto=False recusou=False Sim, a estação tem acesso à plataforma por elevador.
   3 agente        acerto=False recusou=False Sim, a estação tem acesso à plataforma por elevador.
ERRO 3 inversão de "exclusivo"

texto_30: uma pessoa cega pode usar o táxi acessi

## 7. Correções medidas: `agente_v2` (C1, C1b, C2, C3, C4), sem alterar o `agente` original

- **C1 (`gtfs.py`, modo `v2`):** casa só `stop_name` (não `stop_desc`); um local casa com o **grupo** de paradas mais denso (paradas casadas encadeadas a ≤ 300 m), com prioridade para nomes com "Terminal"/"Estação" quando a pergunta tiver a palavra. Ajuste durante a implementação: só "grupo mais denso" escolhia a avenida em vez da estação e quebrava o teste `Tucuruvi -> Jabaquara`; por isso, entre grupos, vem primeiro o que tem uma parada de nome **igual** ao pedido.
- **C1b:** normaliza abreviações do GTFS antes de casar (Sto.→santo, Sta.→santa, Pq.→parque, Vl.→vila, Jd.→jardim, Term.→terminal, Est.→estacao). Corrige o efeito colateral do C1 no `rota_09` ("Terminal Sto. Amaro").
- **C2 (barreira 3):** `citacao_invalida` → recusa.
- **C3:** roteador escolhe "nenhuma" mas o top-1 da busca é ≥ limiar (0,8537) → `buscar_base`.
- **C4:** se `recusou=true`, `resposta` vira "Não consta na base de conhecimento." (mantém "Não tenho previsão em tempo real agora.", que já é uma recusa).

Resultados: `agente_v2.jsonl` (versão final, C1+C1b+C2+C3+C4) e `agente_v2_c1_c3.jsonl` (a execução anterior, sem C1b e C4). O gabarito não foi alterado.

In [9]:
v2 = carregar('agente_v2'); v2_ant = carregar('agente_v2_c1_c3')
v2e, v2_ante = av.reavaliar(v2, por_id), av.reavaliar(v2_ant, por_id)
cmp = {'3 agente (original)': estrito['agente'], '3b v2 (C1-C3)': v2_ante, '3b v2 final (C1b+C4)': v2e}
print('MÉTRICA ESTRITA'); print(pd.DataFrame({k: av.resumo(v) for k, v in cmp.items()}).T[cols].to_string())
for t in ('texto', 'rota', 'nao_consta'):
    print(f'\n{t.upper()}')
    print(pd.DataFrame({k: av.resumo([r for r in v if r['tipo'] == t]) for k, v in cmp.items()}).T[cols].to_string())
print('\nv2 final com a métrica antiga (para conferir o efeito do C4):'); print(pd.Series(av.resumo(v2)).to_string())
print('divergências recusou × texto no v2 final:', av.divergencias(v2))

MÉTRICA ESTRITA
                      acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
3 agente (original)     0.7460      0.1587          0.8667        0.1395          1.0           0.6667              1.87
3b v2 (C1-C3)           0.8095      0.1111          0.9333        0.1163          1.0           0.8000              3.57
3b v2 final (C1b+C4)    0.8413      0.1111          1.0000        0.0698          1.0           0.8000              3.60

TEXTO
                      acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
3 agente (original)     0.6667      0.1667             NaN        0.1667          1.0           0.6667              2.03
3b v2 (C1-C3)           0.7333      0.2000             NaN        0.0667          1.0           0.8000              5.51
3b v2 final (C1b+C4)    0.7333      0.2000             NaN        0.0667          1.0           0.8000              5.47

ROTA
   

### Tabela: erro selecionado × antes × depois (inclui o `rota_09` e o efeito do C4)

In [10]:
antes = {r['id']: r for r in estrito['agente']}
meio = {r['id']: r for r in v2_ante}
final = {r['id']: r for r in v2e}
grupos = [
    ('1 busca', ['texto_19']), ('2 id inventado (C2)', ['nao_consta_01']), ('3 inversão de "exclusivo"', ['texto_30']),
    ('4 falsos positivos do GTFS (C1)', ['rota_04', 'rota_08', 'rota_12']), ('5 roteador -> nenhuma (C3)', ['texto_23', 'texto_25', 'texto_29', 'rota_06', 'rota_14']),
    ('extra: extração (C1)', ['rota_10']), ('efeito colateral do C1 (corrigido pela C1b)', ['rota_09']),
    ('recusou=true com resposta (C4)', ['nao_consta_12']), ('ruído de execução', ['texto_16']),
]
def curta(r):
    s = r['saida']
    return ('[recusa] ' if r['metricas']['recusou'] else '') + s['resposta'].replace('\n', ' ')[:70] + f" ({s['ferramenta']})"
def rotulo(r): return 'ACERTA' if r['metricas']['acerto'] else 'erra'
linhas = []
for nome, ids in grupos:
    for i in ids:
        a, m, f = antes[i], meio[i], final[i]
        resultado = 'corrigido' if (f['metricas']['acerto'] and not a['metricas']['acerto']) else 'igual' if f['metricas']['acerto'] == a['metricas']['acerto'] else 'PIOROU'
        linhas.append({'erro': nome, 'caso': i, 'antes': rotulo(a), 'C1-C3': rotulo(m), 'final': rotulo(f), 'resultado': resultado,
                       'resposta final': curta(f)})
tab = pd.DataFrame(linhas)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 90)
print(tab[['erro', 'caso', 'antes', 'C1-C3', 'final', 'resultado']].to_string(index=False))
print('\n' + tab['resultado'].value_counts().to_string())
print('\nRespostas (antes -> final):')
for l in linhas:
    print(f"  {l['caso']:<14} antes : {curta(antes[l['caso']])}")
    print(f"  {'':<14} final : {l['resposta final']}")

                                       erro          caso  antes  C1-C3  final resultado
                                    1 busca      texto_19   erra   erra   erra     igual
                        2 id inventado (C2) nao_consta_01   erra ACERTA ACERTA corrigido
                  3 inversão de "exclusivo"      texto_30   erra   erra   erra     igual
            4 falsos positivos do GTFS (C1)       rota_04   erra ACERTA ACERTA corrigido
            4 falsos positivos do GTFS (C1)       rota_08 ACERTA   erra ACERTA     igual
            4 falsos positivos do GTFS (C1)       rota_12   erra ACERTA ACERTA corrigido
                 5 roteador -> nenhuma (C3)      texto_23   erra ACERTA ACERTA corrigido
                 5 roteador -> nenhuma (C3)      texto_25   erra ACERTA ACERTA corrigido
                 5 roteador -> nenhuma (C3)      texto_29   erra ACERTA ACERTA corrigido
                 5 roteador -> nenhuma (C3)       rota_06   erra   erra   erra     igual
                 5 ro

### Todos os casos em que o acerto mudou, do agente original para o v2 final

In [11]:
for i, a in antes.items():
    ma, mf = a['metricas']['acerto'], final[i]['metricas']['acerto']
    if ma != mf:
        s = final[i]['saida']
        print(f"{i:<14} {'ACERTA' if ma else 'erra':<7} -> {'ACERTA' if mf else 'erra':<7} {s['ferramenta']:<15} barreira={s['barreira']!s:<17} {s['resposta'][:80]}")
print()
print('barreira 3 (citacao_invalida):', [i for i, r in final.items() if r['saida']['barreira'] == 'citacao_invalida'])
print('fallback do roteador (C3)   :', [i for i, r in final.items() if r['saida'].get('fallback_roteador')])
print('C4 reescreveu a resposta em :', [i for i, r in final.items() if 'resposta_original' in r['saida']])
print('alucinações do v2 final     :', [i for i, r in final.items() if r['metricas']['alucinacao']])

texto_16       ACERTA  -> erra    buscar_base     barreira=None              Sempre que identificar uma pessoa cega no ponto, pare e informe qual a linha que
texto_23       erra    -> ACERTA  buscar_base     barreira=None              Você precisa levar o CPF (original e cópia), RG ou certidão de nascimento (origi
texto_25       erra    -> ACERTA  buscar_base     barreira=None              As frotas de empresas de táxi devem reservar 10% de seus veículos acessíveis à p
texto_29       erra    -> ACERTA  buscar_base     barreira=None              35 estações da CPTM já possuem condições de acesso às pessoas com deficiência ou
rota_04        erra    -> ACERTA  consultar_gtfs  barreira=None              Você pode usar o Trem Estudantes da Linha L11 da CPTM ou o Metrô L3 sentido Cori
rota_07        ACERTA  -> erra    consultar_gtfs  barreira=None              Não há linhas de trem disponíveis para você. As opções são ônibus.
rota_10        erra    -> ACERTA  consultar_gtfs  barreira=ferrame

## 8. Validação (`avaliacao/casos_validacao.jsonl`, 12 casos escritos à parte)

Resultados de `agente`, do v2 só com C1-C3 e do v2 final, cada um separado (métrica estrita).

In [12]:
va = av.reavaliar(carregar('validacao_agente'), por_id)
vb = av.reavaliar(carregar('validacao_agente_v2_c1_c3'), por_id)
vc = av.reavaliar(carregar('validacao_agente_v2'), por_id)
for nome, regs in (('agente (original)', va), ('agente_v2 (C1-C3)', vb), ('agente_v2 final (C1b+C4)', vc)):
    print(nome); print(pd.DataFrame({'geral': av.resumo(regs), **av.resumo_por_tipo(regs)}).T[cols].to_string()); print()
print('Caso a caso (acerto):')
for a, b, c in zip(va, vb, vc):
    print(f"  {a['id']:<18} agente={rotulo(a):<7} v2(C1-C3)={rotulo(b):<7} v2 final={rotulo(c):<7} | {c['saida']['ferramenta']:<15} args={c['saida']['argumentos']} | {c['saida']['resposta'][:75]}")

agente (original)
            acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
geral         0.6667      0.2500             0.8        0.1429          1.0              1.0              2.04
texto         0.7500      0.2500             NaN        0.0000          1.0              1.0              2.48
rota          0.3333      0.3333             NaN        0.3333          1.0              NaN              1.37
nao_consta    0.8000      0.2000             0.8           NaN          1.0              NaN              2.09

agente_v2 (C1-C3)
            acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
geral         0.6667      0.2500             0.8        0.1429          1.0              1.0              1.78
texto         0.7500      0.2500             NaN        0.0000          1.0              1.0              2.62
rota          0.3333      0.3333             NaN        0.3333          1.0

### Leitura da validação

As correções resolvem **os padrões que atacaram** (falsos positivos do GTFS, roteador "nenhuma", citação inválida, `recusou=true` com resposta), mas a validação mostra erros de **tipos novos**, que nenhuma correção cobre:

- **`val_rota_01` (Luz → Fradique Coutinho):** o roteador extraiu origem "Fradique Coutinho" e destino "Vila São João" (trocou a ordem e trouxe um destino que não está na pergunta). A ferramenta não achou e o agente recusou (falsa recusa).
- **`val_texto_01` (Anhangabaú tem elevador?):** resposta parcial: diz só "elevador" para a plataforma e omite que a entrada é por rampa (o trecho tem as duas colunas).
- **`val_rota_03` (Chácara Klabin → Santa Cruz):** diz que existe trem direto, sem citar a linha 5 (a resposta é correta na substância, o acerto automático exige o código da linha).

Esses erros entram em `avaliacao/erros.md` como "erros encontrados na validação".

## 9. Duas ressalvas de medição

**(a) Variação entre execuções.** O mesmo `agente` (mesmo código, `temperatura=0`, `seed=42`) foi rodado uma segunda vez (`resultados/agente_rerun.jsonl`). O Ollama não é 100% determinístico, e isso entra como ruído nos números (cerca de 1 caso).

In [13]:
rerun = carregar('agente_rerun')
orig = {r['id']: r for r in bruto['agente']}
mudou_texto = [r['id'] for r in rerun if r['saida']['resposta'] != orig[r['id']]['saida']['resposta']]
mudou_acerto = [(r['id'], orig[r['id']]['metricas']['acerto'], r['metricas']['acerto']) for r in rerun if r['metricas']['acerto'] != orig[r['id']]['metricas']['acerto']]
print('texto da resposta diferente na reexecução:', len(mudou_texto), 'de', len(rerun))
print('acerto diferente:', mudou_acerto)
print('acerto geral (métrica antiga): original', av.resumo(bruto['agente'])['acuracia'], '| reexecução', av.resumo(rerun)['acuracia'])
mesmo_caminho = [r['id'] for r in rerun if r['saida']['recuperados'] == orig[r['id']]['saida']['recuperados'] and r['saida']['ferramenta'] == orig[r['id']]['saida']['ferramenta']]
print('mesma ferramenta e mesmos trechos recuperados em', len(mesmo_caminho), 'casos (a diferença vem só da geração do LLM)')

texto da resposta diferente na reexecução: 16 de 63
acerto diferente: [('texto_16', True, False)]
acerto geral (métrica antiga): original 0.7619 | reexecução 0.746
mesma ferramenta e mesmos trechos recuperados em 63 casos (a diferença vem só da geração do LLM)


**(b) Divergência entre o campo `recusou` e o texto** (o que a métrica estrita corrige): resumo por arquivo de resultados.

In [14]:
for n in ['sem_contexto', 'sem_contexto_instr', 'rag', 'agente', 'agente_v2_c1_c3', 'agente_v2', 'validacao_agente', 'validacao_agente_v2_c1_c3', 'validacao_agente_v2']:
    print(f"{n:<28} {[(d['id'], d['recusou']) for d in av.divergencias(carregar(n))]}")

sem_contexto                 [('nao_consta_06', False), ('nao_consta_08', False)]
sem_contexto_instr           []
rag                          [('nao_consta_12', True)]
agente                       [('nao_consta_12', True)]
agente_v2_c1_c3              [('nao_consta_12', True)]
agente_v2                    []
validacao_agente             [('val_nao_consta_05', True)]
validacao_agente_v2_c1_c3    [('val_nao_consta_05', True)]
validacao_agente_v2          []
